# SMAX checkpoint evaluation

This notebook evaluates the curated `SMAX_checkpts/` tree with deterministic rollouts and reports paper-table win rates at the training-seed level. It expects each map/constraint/model group to contain the five configured training seeds `{1, 2, 31, 32, 42}` and uses final checkpoints by default.


In [3]:
import importlib
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "main.py").is_file() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the pci_marl repo root from the current notebook working directory.")


ROOT = resolve_repo_root()
ARTIFACT_ROOT = ROOT
FIGURES_DIR = ARTIFACT_ROOT / "figures"
CHECKPOINTS_DIR = ARTIFACT_ROOT / "runs/checkpoints_smax"

for path in (ROOT, FIGURES_DIR):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

command_buffer_flag = "--xla_gpu_enable_command_buffer="
existing_xla_flags = os.environ.get("XLA_FLAGS", "")
if command_buffer_flag not in existing_xla_flags:
    os.environ["XLA_FLAGS"] = f"{existing_xla_flags} {command_buffer_flag}".strip()
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

import tab2_smax_checkpoint_eval_utils as scu
scu = importlib.reload(scu)

bootstrap_project = scu.bootstrap_project
discover_runs = scu.discover_runs
evaluate_map = scu.evaluate_map
evaluate_map_constraint = scu.evaluate_map_constraint
format_cross_map_latex = scu.format_cross_map_latex
ordered_map_names = scu.ordered_map_names
ordered_model_labels = scu.ordered_model_labels
save_summary_tables = scu.save_summary_tables
save_full_evaluation_bundle = scu.save_full_evaluation_bundle
clear_runtime_caches = scu.clear_runtime_caches

DEVICE = "cuda"
CHECKPOINT_CHOICE = "final"  # or "best"
DUPLICATE_POLICY = "latest"
FORCE_REEVALUATE = False  # False: reuse cached runs and evaluate only uncached/new checkpoints

EVAL_SEEDS = list(range(10001, 10065))
EVAL_EPISODES = 1
EVAL_SEED_BATCH_SIZE = 16
EXPECTED_TRAIN_SEEDS = {1, 2, 31, 32, 42}
EXPECTED_TRAIN_SEED_TUPLE = tuple(sorted(EXPECTED_TRAIN_SEEDS))
MAP_RESULTS = {}

if FORCE_REEVALUATE:
    clear_runtime_caches()

bootstrap_project(DEVICE)
runs_df = discover_runs(CHECKPOINTS_DIR, project_filter=None)
if runs_df.empty:
    raise RuntimeError(f"No runs were discovered under {CHECKPOINTS_DIR}")

checkpoint_column = f"{CHECKPOINT_CHOICE}_checkpoint"
missing_checkpoint_runs = runs_df[~runs_df[checkpoint_column].map(lambda path: Path(path).exists())].copy()
if not missing_checkpoint_runs.empty:
    print(f"Skipping {len(missing_checkpoint_runs)} runs without {CHECKPOINT_CHOICE} checkpoints.")
    display(missing_checkpoint_runs[["map_name", "constraint_key", "model_label", "seed", "run_dir", checkpoint_column]])
runs_df = runs_df[runs_df[checkpoint_column].map(lambda path: Path(path).exists())].reset_index(drop=True)

seed_sets = (
    runs_df.groupby(["map_name", "constraint_key", "model_label"])["seed"]
    .agg(lambda values: tuple(sorted(set(int(value) for value in values))))
    .reset_index(name="train_seed_set")
)
incomplete_seed_sets = seed_sets[seed_sets["train_seed_set"] != EXPECTED_TRAIN_SEED_TUPLE]
if not incomplete_seed_sets.empty:
    raise AssertionError(
        "SMAX_checkpts is missing configured training seeds:\n"
        + incomplete_seed_sets.to_string(index=False)
    )

expected_run_count = len(seed_sets) * len(EXPECTED_TRAIN_SEEDS)
if len(runs_df) != expected_run_count:
    raise AssertionError(f"Expected {expected_run_count} evaluable runs, found {len(runs_df)}")


In [4]:
# Select Maps and Models

TARGET_MAPS = ordered_map_names(runs_df["map_name"].dropna().unique().tolist())
TARGET_MODELS = ordered_model_labels(runs_df["model_label"].dropna().unique().tolist())

print(f"Repo root: {ROOT}")
print(f"Checkpoint root: {CHECKPOINTS_DIR}")
print(f"Loaded {len(runs_df)} evaluation runs from {CHECKPOINTS_DIR}")
print(f"Expected training seeds per group: {EXPECTED_TRAIN_SEED_TUPLE}")
print(f"Maps: {TARGET_MAPS}")
print(f"Models: {TARGET_MODELS}")
display(runs_df.groupby(["map_name", "constraint_label", "model_label"]).size().unstack(fill_value=0))
print(f"Use cached run evaluations: {not FORCE_REEVALUATE}")


Repo root: /home/chamjin/pci_marl
Checkpoint root: /home/chamjin/pci_marl/runs/checkpoints_smax
Loaded 300 evaluation runs from /home/chamjin/pci_marl/runs/checkpoints_smax
Expected training seeds per group: (1, 2, 31, 32, 42)
Maps: ['3m', '2s3z', '6h_vs_8z', '3s5z', 'smacv2_5_units', 'smacv2_10_units']
Models: ['ippo', 'pci-ippo', 'mappo', 'pci-mappo']


model_label                       ippo  mappo  pci-ippo  pci-mappo
map_name        constraint_label                                  
2s3z            tar_w=3              5      5         5          5
                tar_w=4              5      5         5          5
                vanilla              5      5         5          5
3m              tar_w=2              5      5         5          5
                vanilla              5      5         5          5
3s5z            tar_w=6              5      5         5          5
                vanilla              5      5         5          5
6h_vs_8z        tar_w=6              5      5         5          5
                vanilla              5      5         5          5
smacv2_10_units tar_w=6              5      5         5          5
                tar_w=8              5      5         5          5
                vanilla              5      5         5          5
smacv2_5_units  tar_w=3              5      5         5          5
                tar_w=4              5      5         5          5
                vanilla              5      5         5          5

Use cached run evaluations: True


In [5]:
SMOKE_EVAL_SEEDS = EVAL_SEEDS[:2]

smoke_row = runs_df.sort_values(["map_name", "constraint_key", "model_label", "seed", "run_dir"]).iloc[0]
SMOKE_MAP = smoke_row["map_name"]
SMOKE_CONSTRAINT = smoke_row["constraint_key"]
SMOKE_MODEL = smoke_row["model_label"]
SMOKE_SEED = int(smoke_row["seed"])

smoke_runs = runs_df[
    (runs_df["map_name"] == SMOKE_MAP)
    & (runs_df["constraint_key"] == SMOKE_CONSTRAINT)
    & (runs_df["model_label"] == SMOKE_MODEL)
    & (runs_df["seed"] == SMOKE_SEED)
].copy()

display(smoke_runs[["map_name", "constraint_key", "model_label", "seed", "wandb_run_name", "run_dir"]])

smoke_result = evaluate_map_constraint(
    smoke_runs,
    SMOKE_MAP,
    SMOKE_CONSTRAINT,
    checkpoint_choice=CHECKPOINT_CHOICE,
    device=DEVICE,
    eval_episodes=EVAL_EPISODES,
    eval_seeds=SMOKE_EVAL_SEEDS,
    eval_seed_batch_size=min(EVAL_SEED_BATCH_SIZE, len(SMOKE_EVAL_SEEDS)),
    duplicate_policy=DUPLICATE_POLICY,
    use_cache=not FORCE_REEVALUATE,
)
display(smoke_result["run_results"])
display(smoke_result["seed_summary"])
display(smoke_result["model_summary"])


,map_name,constraint_key,model_label,seed,wandb_run_name,run_dir
0,2s3z,tar_w=3,ippo,1,smax_2s3z_tar_w=3_hard_lr0.004,/home/chamjin/pci_marl/runs/checkpoints_smax/2...


[SMAXWrapper] '2s3z': 5 allies vs 5 enemies
[SMAXBatchWrapper] '2s3z': 5v5, n_envs=2
[SMAXBatchWrapper] allowed_targets (full source-level damage mask, window=3): {0: [0, 1, 2], 1: [1, 2, 3], 2: [2, 3, 4], 3: [3, 4, 0], 4: [4, 0, 1]}
[SMAXBatchWrapper] obs_dim=127, action_dim=10
[SMAXBatchWrapper] 5v5 × 2 emvs
[INFO] add_agent_id enabled: base_obs_dim=127, effective_obs_dim=132


/tmp/ipykernel_2407553/3843037941.py:18: UserWarning: std_success_rate is defined over training seeds. Filled 0.0 for rows with n_seeds=1.
map_name constraint_key model_label  n_seeds
    2s3z        tar_w=3        ippo        1
  smoke_result = evaluate_map_constraint(


,eval_success_rate,eval_mean_reward,eval_std_reward,eval_mean_length,eval_min_reward,eval_max_reward,run_dir,metadata_path,project,map_name,...,constraint_label,algorithm_type,model_key,model_label,seed,train_seed,eval_seed,checkpoint_choice,checkpoint_path,num_episodes
0,1.0,2.00,0.0,64.0,2.00,2.00,/home/chamjin/pci_marl/runs/checkpoints_smax/2...,/home/chamjin/pci_marl/runs/checkpoints_smax/2...,pci-marl-smax-jax-v2,2s3z,...,tar_w=3,jax_ippo,ippo,ippo,1,1,10001,final,/home/chamjin/pci_marl/runs/checkpoints_smax/2...,1
1,0.0,0.82,0.0,75.0,0.82,0.82,/home/chamjin/pci_marl/runs/checkpoints_smax/2...,/home/chamjin/pci_marl/runs/checkpoints_smax/2...,pci-marl-smax-jax-v2,2s3z,...,tar_w=3,jax_ippo,ippo,ippo,1,1,10002,final,/home/chamjin/pci_marl/runs/checkpoints_smax/2...,1


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,2s3z,tar_w=3,tar_w=3,ippo,1,0.5,1.41,69.5,1,2


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,2s3z,tar_w=3,tar_w=3,ippo,0.5,0.0,1.41,NaN,69.5,NaN,1,1,2


## Full evaluation

This cell evaluates each target map with a shared eval-seed set and preserves the aggregation shape:

- `run_results`: each train seed x eval seed result
- `seed_summary`: each train seed averaged over eval seeds
- `model_summary`: mean $\pm$ std over training seeds


In [6]:
MAP_RESULTS = {}
for map_name in TARGET_MAPS:
    result = evaluate_map(
        runs_df,
        map_name,
        checkpoint_choice=CHECKPOINT_CHOICE,
        device=DEVICE,
        eval_episodes=EVAL_EPISODES,
        eval_seeds=EVAL_SEEDS,
        eval_seed_batch_size=EVAL_SEED_BATCH_SIZE,
        model_order=TARGET_MODELS,
        duplicate_policy=DUPLICATE_POLICY,
        use_cache=not FORCE_REEVALUATE,
    )
    if result["model_summary"].empty:
        print(f"Skipping {map_name}: no discovered runs")
        continue
    MAP_RESULTS[map_name] = result
    print(f"\n=== {map_name} ===")
    display(result["seed_summary"])
    display(result["model_summary"])
    print(result["latex"])


[SMAXWrapper] '3m': 3 allies vs 3 enemies
[SMAXBatchWrapper] '3m': 3v3, n_envs=16
[SMAXBatchWrapper] obs_dim=75, action_dim=8
[SMAXBatchWrapper] 3v3 × 16 emvs
[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78
[SMAXWrapper] '3m': 3 allies vs 3 enemies
[SMAXBatchWrapper] '3m': 3v3, n_envs=16
[SMAXBatchWrapper] obs_dim=75, action_dim=8
[SMAXBatchWrapper] 3v3 × 16 emvs
[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78
[SMAXWrapper] '3m': 3 allies vs 3 enemies
[SMAXBatchWrapper] '3m': 3v3, n_envs=16
[SMAXBatchWrapper] obs_dim=75, action_dim=8
[SMAXBatchWrapper] 3v3 × 16 emvs
[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78
[SMAXWrapper] '3m': 3 allies vs 3 enemies
[SMAXBatchWrapper] '3m': 3v3, n_envs=16
[SMAXBatchWrapper] obs_dim=75, action_dim=8
[SMAXBatchWrapper] 3v3 × 16 emvs
[INFO] add_agent_id enabled: base_obs_dim=75, effective_obs_dim=78
[SMAXWrapper] '3m': 3 allies vs 3 enemies
[SMAXBatchWrapper] '3m': 3v3, n_envs=16
[SMAXBatchWrap

,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,ippo,1,0.875000,1.863542,19.156250,1,64
1,3m,vanilla,vanilla,ippo,2,0.890625,1.863542,15.906250,1,64
2,3m,vanilla,vanilla,ippo,31,0.937500,1.909375,21.843750,1,64
3,3m,vanilla,vanilla,ippo,32,1.000000,2.000000,20.828125,1,64
4,3m,vanilla,vanilla,ippo,42,0.812500,1.783333,16.296875,1,64
5,3m,vanilla,vanilla,pci-ippo,1,0.968750,1.963542,29.812500,1,64
6,3m,vanilla,vanilla,pci-ippo,2,0.984375,1.983333,26.078125,1,64
7,3m,vanilla,vanilla,pci-ippo,31,0.984375,1.980208,25.140625,1,64
8,3m,vanilla,vanilla,pci-ippo,32,0.968750,1.963542,24.687500,1,64
9,3m,vanilla,vanilla,pci-ippo,42,0.968750,1.960417,25.531250,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,ippo,0.903125,0.070226,1.883958,0.079169,18.806250,2.652539,5,5,64
1,3m,vanilla,vanilla,pci-ippo,0.975000,0.008558,1.970208,0.010689,26.250000,2.056068,5,5,64
2,3m,vanilla,vanilla,mappo,0.875000,0.052987,1.856667,0.064975,15.593750,0.944505,5,5,64
3,3m,vanilla,vanilla,pci-mappo,0.931250,0.028384,1.920625,0.034528,30.665625,0.996000,5,5,64
4,3m,tar_w=2,tar_w=2,ippo,0.409375,0.067568,1.259167,0.097136,19.715625,6.785024,5,5,64
5,3m,tar_w=2,tar_w=2,pci-ippo,0.856250,0.069351,1.825208,0.094796,36.409375,1.621593,5,5,64
6,3m,tar_w=2,tar_w=2,mappo,0.556250,0.246459,1.462292,0.298700,17.537500,4.178740,5,5,64
7,3m,tar_w=2,tar_w=2,pci-mappo,0.887500,0.033874,1.862917,0.045283,39.493750,1.922593,5,5,64


\begin{table}[htbp]
\vspace{-1em}
\caption{Test win rates (\%) on SMAX maps. $\pm$ denotes one standard deviation over available training seeds. The best results are highlighted in bold. The sign of \(\Delta\) and colored values indicate performance improvements (blue) or degradations (red) over their respective baselines.}
\label{tab:smax}
\centering
\resizebox{0.9\textwidth}{!}{%

\begin{tabular}{c c c c c c c c}
\toprule
Map & Target Constraint & IPPO & PCI-IPPO* & \(\Delta\) & MAPPO & PCI-MAPPO* & \(\Delta\)\\
\midrule
    3m & Vanilla & 90.3 \scriptsize$\pm$ 7.0 & \textbf{97.5} \scriptsize$\pm$ 0.9 & \textbf{\textcolor{blue}{7.2}} & 87.5 \scriptsize$\pm$ 5.3 & 93.1 \scriptsize$\pm$ 2.8 & \textbf{\textcolor{blue}{5.6}} \\
     & Hard & 40.9 \scriptsize$\pm$ 6.8 & 85.6 \scriptsize$\pm$ 6.9 & \textbf{\textcolor{blue}{44.7}} & 55.6 \scriptsize$\pm$ 24.6 & \textbf{88.8} \scriptsize$\pm$ 3.4 & \textbf{\textcolor{blue}{33.1}} \\
\bottomrule
\end{tabular}}
\end{table}
[SMAXWrapper] '2s3z'

,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,2s3z,vanilla,vanilla,ippo,1,1.000000,2.000000,52.843750,1,64
1,2s3z,vanilla,vanilla,ippo,2,1.000000,2.000000,50.750000,1,64
2,2s3z,vanilla,vanilla,ippo,31,1.000000,2.000000,52.984375,1,64
3,2s3z,vanilla,vanilla,ippo,32,0.921875,1.910600,61.609375,1,64
4,2s3z,vanilla,vanilla,ippo,42,1.000000,2.000000,52.562500,1,64
5,2s3z,vanilla,vanilla,pci-ippo,1,1.000000,2.000000,50.937500,1,64
6,2s3z,vanilla,vanilla,pci-ippo,2,0.953125,1.950605,63.593750,1,64
7,2s3z,vanilla,vanilla,pci-ippo,31,1.000000,2.000000,49.656250,1,64
8,2s3z,vanilla,vanilla,pci-ippo,32,1.000000,2.000000,61.859375,1,64
9,2s3z,vanilla,vanilla,pci-ippo,42,0.984375,1.983867,55.859375,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,2s3z,vanilla,vanilla,ippo,0.984375,0.034939,1.982120,3.998079e-02,54.150000,4.265777,5,5,64
1,2s3z,vanilla,vanilla,pci-ippo,0.987500,0.020373,1.986894,2.145534e-02,56.381250,6.268221,5,5,64
2,2s3z,vanilla,vanilla,mappo,0.996875,0.006988,1.996750,7.267219e-03,57.721875,0.272297,5,5,64
3,2s3z,vanilla,vanilla,pci-mappo,1.000000,0.000000,2.000000,1.848623e-08,51.753125,2.106375,5,5,64
4,2s3z,tar_w=3,tar_w=3,ippo,0.734375,0.228307,1.704916,2.569314e-01,70.846875,3.746261,5,5,64
5,2s3z,tar_w=3,tar_w=3,pci-ippo,0.781250,0.436802,1.643997,7.406232e-01,55.440625,11.591767,5,5,64
6,2s3z,tar_w=3,tar_w=3,mappo,0.728125,0.056983,1.700900,6.383686e-02,69.893750,1.851493,5,5,64
7,2s3z,tar_w=3,tar_w=3,pci-mappo,0.993750,0.008558,1.993321,9.146680e-03,57.156250,0.843099,5,5,64
8,2s3z,tar_w=4,tar_w=4,ippo,0.790625,0.131008,1.765494,1.494256e-01,71.821875,7.357505,5,5,64
9,2s3z,tar_w=4,tar_w=4,pci-ippo,0.953125,0.048159,1.947800,5.414500e-02,62.915625,3.260249,5,5,64


... (9 outputs left out)

\begin{table}[htbp]
\vspace{-1em}
\caption{Test win rates (\%) on SMAX maps. $\pm$ denotes one standard deviation over available training seeds. The best results are highlighted in bold. The sign of \(\Delta\) and colored values indicate performance improvements (blue) or degradations (red) over their respective baselines.}
\label{tab:smax}
\centering
\resizebox{0.9\textwidth}{!}{%

\begin{tabular}{c c c c c c c c}
\toprule
Map & Target Constraint & IPPO & PCI-IPPO* & \(\Delta\) & MAPPO & PCI-MAPPO* & \(\Delta\)\\
\midrule
    smacv2\_5\_units & Vanilla & 67.2 \scriptsize$\pm$ 2.5 & 77.8 \scriptsize$\pm$ 3.9 & \textbf{\textcolor{blue}{10.6}} & 73.8 \scriptsize$\pm$ 3.7 & \textbf{80.9} \scriptsize$\pm$ 2.3 & \textbf{\textcolor{blue}{7.2}} \\
     & Medium & 46.9 \scriptsize$\pm$ 2.2 & 65.9 \scriptsize$\pm$ 5.8 & \textbf{\textcolor{blue}{19.1}} & 57.2 \scriptsize$\pm$ 4.5 & \textbf{67.8} \scriptsize$\pm$ 4.1 & \textbf{\textcolor{blue}{10.6}} \\
     & Hard & 23.8 \scriptsize$\pm$ 3.0 & 4

,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,smacv2_10_units,vanilla,vanilla,ippo,1,0.421875,1.241164,55.796875,1,64
1,smacv2_10_units,vanilla,vanilla,ippo,2,0.328125,1.082556,54.593750,1,64
2,smacv2_10_units,vanilla,vanilla,ippo,31,0.296875,1.000940,53.062500,1,64
3,smacv2_10_units,vanilla,vanilla,ippo,32,0.468750,1.260082,49.828125,1,64
4,smacv2_10_units,vanilla,vanilla,ippo,42,0.312500,1.062235,53.453125,1,64
5,smacv2_10_units,vanilla,vanilla,pci-ippo,1,0.437500,1.256102,53.703125,1,64
6,smacv2_10_units,vanilla,vanilla,pci-ippo,2,0.500000,1.325569,52.843750,1,64
7,smacv2_10_units,vanilla,vanilla,pci-ippo,31,0.484375,1.290604,53.187500,1,64
8,smacv2_10_units,vanilla,vanilla,pci-ippo,32,0.484375,1.318239,51.375000,1,64
9,smacv2_10_units,vanilla,vanilla,pci-ippo,42,0.421875,1.232880,55.453125,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,smacv2_10_units,vanilla,vanilla,ippo,0.365625,0.075422,1.129395,0.114866,53.346875,2.237798,5,5,64
1,smacv2_10_units,vanilla,vanilla,pci-ippo,0.465625,0.033874,1.284679,0.039791,53.312500,1.476831,5,5,64
2,smacv2_10_units,vanilla,vanilla,mappo,0.525000,0.047650,1.374029,0.059744,41.403125,0.690459,5,5,64
3,smacv2_10_units,vanilla,vanilla,pci-mappo,0.628125,0.020373,1.504114,0.032071,40.337500,1.401842,5,5,64
4,smacv2_10_units,tar_w=6,tar_w=6,ippo,0.084375,0.088802,0.724185,0.164977,47.621875,3.350741,5,5,64
5,smacv2_10_units,tar_w=6,tar_w=6,pci-ippo,0.112500,0.090030,0.794029,0.139543,51.328125,0.987470,5,5,64
6,smacv2_10_units,tar_w=6,tar_w=6,mappo,0.181250,0.045015,0.951850,0.058097,45.190625,1.164005,5,5,64
7,smacv2_10_units,tar_w=6,tar_w=6,pci-mappo,0.293750,0.086574,1.105341,0.105191,44.950000,1.359797,5,5,64
8,smacv2_10_units,tar_w=8,tar_w=8,ippo,0.040625,0.028384,0.642862,0.099635,47.059375,3.205837,5,5,64
9,smacv2_10_units,tar_w=8,tar_w=8,pci-ippo,0.140625,0.052987,0.856302,0.085141,53.118750,2.260220,5,5,64


\begin{table}[htbp]
\vspace{-1em}
\caption{Test win rates (\%) on SMAX maps. $\pm$ denotes one standard deviation over available training seeds. The best results are highlighted in bold. The sign of \(\Delta\) and colored values indicate performance improvements (blue) or degradations (red) over their respective baselines.}
\label{tab:smax}
\centering
\resizebox{0.9\textwidth}{!}{%

\begin{tabular}{c c c c c c c c}
\toprule
Map & Target Constraint & IPPO & PCI-IPPO* & \(\Delta\) & MAPPO & PCI-MAPPO* & \(\Delta\)\\
\midrule
    smacv2\_10\_units & Vanilla & 36.6 \scriptsize$\pm$ 7.5 & 46.6 \scriptsize$\pm$ 3.4 & \textbf{\textcolor{blue}{10.0}} & 52.5 \scriptsize$\pm$ 4.8 & \textbf{62.8} \scriptsize$\pm$ 2.0 & \textbf{\textcolor{blue}{10.3}} \\
     & Medium & 4.1 \scriptsize$\pm$ 2.8 & 14.1 \scriptsize$\pm$ 5.3 & \textbf{\textcolor{blue}{10.0}} & 36.9 \scriptsize$\pm$ 6.5 & \textbf{60.3} \scriptsize$\pm$ 3.2 & \textbf{\textcolor{blue}{23.4}} \\
     & Hard & 8.4 \scriptsize$\pm$ 8.9 & 1

In [7]:
for map_name, result in MAP_RESULTS.items():
    run_df = result["run_results"].copy()
    seed_summary = result["seed_summary"].copy()
    model_summary = result["model_summary"].copy()

    eval_seed_sets = (
        run_df.groupby(["constraint_key", "model_label", "train_seed"])["eval_seed"]
        .agg(lambda values: tuple(sorted(set(int(v) for v in values))))
        .reset_index(name="eval_seed_set")
    )
    method_seed_counts = eval_seed_sets.groupby(["constraint_key", "model_label"])["eval_seed_set"].nunique()
    assert (method_seed_counts == 1).all(), f"Inconsistent eval_seed sets detected for {map_name}:\n{method_seed_counts}"

    assert not (run_df["train_seed"] == run_df["eval_seed"]).any(), f"train_seed and eval_seed collided for {map_name}"

    expected_seed_rows = (
        run_df.groupby(["constraint_key", "model_label"])["train_seed"]
        .nunique()
        .reset_index(name="expected_rows")
    )
    actual_seed_rows = (
        seed_summary.groupby(["constraint_key", "model_label"])
        .size()
        .reset_index(name="actual_rows")
    )
    row_check = expected_seed_rows.merge(actual_seed_rows, on=["constraint_key", "model_label"], how="left")
    assert (row_check["expected_rows"] == row_check["actual_rows"]).all(), f"seed_summary row mismatch for {map_name}:\n{row_check}"

    expected_std = (
        seed_summary.groupby(["constraint_key", "model_label"])["eval_success_rate"]
        .std()
        .reset_index(name="expected_std_success_rate")
    )
    std_check = model_summary.merge(expected_std, on=["constraint_key", "model_label"], how="left")
    observed = std_check["std_success_rate"].fillna(0.0).to_numpy()
    expected = std_check["expected_std_success_rate"].fillna(0.0).to_numpy()
    assert ((observed - expected).__abs__() < 1.0e-12).all(), f"std_success_rate mismatch for {map_name}:\n{std_check}"

    expected_seed_count = len(EXPECTED_TRAIN_SEEDS)
    assert (model_summary["n_seeds"] == expected_seed_count).all(), (
        f"model_summary should use {expected_seed_count} training seeds for {map_name}:\n"
        f"{model_summary[['constraint_key', 'model_label', 'n_seeds']]}"
    )

    if DUPLICATE_POLICY != "all":
        assert (seed_summary["n_runs"] == 1).all(), f"duplicate runs leaked into seed_summary for {map_name}:\n{seed_summary[['constraint_key', 'model_label', 'seed', 'n_runs']]}"
        assert (model_summary["n_runs"] == expected_seed_count).all(), (
            f"model_summary should have {expected_seed_count} runs per method for {map_name}:\n"
            f"{model_summary[['constraint_key', 'model_label', 'n_runs']]}"
        )

print("Protocol assertions passed: shared eval seeds, no train/eval seed collision, five train seeds per group, and one run per train seed.")


Protocol assertions passed: shared eval seeds, no train/eval seed collision, five train seeds per group, and one run per train seed.


In [8]:
INSPECT_MAP = TARGET_MAPS[0]
if INSPECT_MAP not in MAP_RESULTS:
    raise RuntimeError(f"Run the full evaluation cell first. Missing {INSPECT_MAP}.")

display(MAP_RESULTS[INSPECT_MAP]["seed_summary"])
display(MAP_RESULTS[INSPECT_MAP]["model_summary"])
print(MAP_RESULTS[INSPECT_MAP]["latex"])


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,ippo,1,0.875000,1.863542,19.156250,1,64
1,3m,vanilla,vanilla,ippo,2,0.890625,1.863542,15.906250,1,64
2,3m,vanilla,vanilla,ippo,31,0.937500,1.909375,21.843750,1,64
3,3m,vanilla,vanilla,ippo,32,1.000000,2.000000,20.828125,1,64
4,3m,vanilla,vanilla,ippo,42,0.812500,1.783333,16.296875,1,64
5,3m,vanilla,vanilla,pci-ippo,1,0.968750,1.963542,29.812500,1,64
6,3m,vanilla,vanilla,pci-ippo,2,0.984375,1.983333,26.078125,1,64
7,3m,vanilla,vanilla,pci-ippo,31,0.984375,1.980208,25.140625,1,64
8,3m,vanilla,vanilla,pci-ippo,32,0.968750,1.963542,24.687500,1,64
9,3m,vanilla,vanilla,pci-ippo,42,0.968750,1.960417,25.531250,1,64


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,ippo,0.903125,0.070226,1.883958,0.079169,18.806250,2.652539,5,5,64
1,3m,vanilla,vanilla,pci-ippo,0.975000,0.008558,1.970208,0.010689,26.250000,2.056068,5,5,64
2,3m,vanilla,vanilla,mappo,0.875000,0.052987,1.856667,0.064975,15.593750,0.944505,5,5,64
3,3m,vanilla,vanilla,pci-mappo,0.931250,0.028384,1.920625,0.034528,30.665625,0.996000,5,5,64
4,3m,tar_w=2,tar_w=2,ippo,0.409375,0.067568,1.259167,0.097136,19.715625,6.785024,5,5,64
5,3m,tar_w=2,tar_w=2,pci-ippo,0.856250,0.069351,1.825208,0.094796,36.409375,1.621593,5,5,64
6,3m,tar_w=2,tar_w=2,mappo,0.556250,0.246459,1.462292,0.298700,17.537500,4.178740,5,5,64
7,3m,tar_w=2,tar_w=2,pci-mappo,0.887500,0.033874,1.862917,0.045283,39.493750,1.922593,5,5,64


\begin{table}[htbp]
\vspace{-1em}
\caption{Test win rates (\%) on SMAX maps. $\pm$ denotes one standard deviation over available training seeds. The best results are highlighted in bold. The sign of \(\Delta\) and colored values indicate performance improvements (blue) or degradations (red) over their respective baselines.}
\label{tab:smax}
\centering
\resizebox{0.9\textwidth}{!}{%

\begin{tabular}{c c c c c c c c}
\toprule
Map & Target Constraint & IPPO & PCI-IPPO* & \(\Delta\) & MAPPO & PCI-MAPPO* & \(\Delta\)\\
\midrule
    3m & Vanilla & 90.3 \scriptsize$\pm$ 7.0 & \textbf{97.5} \scriptsize$\pm$ 0.9 & \textbf{\textcolor{blue}{7.2}} & 87.5 \scriptsize$\pm$ 5.3 & 93.1 \scriptsize$\pm$ 2.8 & \textbf{\textcolor{blue}{5.6}} \\
     & Hard & 40.9 \scriptsize$\pm$ 6.8 & 85.6 \scriptsize$\pm$ 6.9 & \textbf{\textcolor{blue}{44.7}} & 55.6 \scriptsize$\pm$ 24.6 & \textbf{88.8} \scriptsize$\pm$ 3.4 & \textbf{\textcolor{blue}{33.1}} \\
\bottomrule
\end{tabular}}
\end{table}


In [9]:
if not MAP_RESULTS:
    raise RuntimeError("Run the full evaluation cell first.")

combined_model_summary = pd.concat(
    [MAP_RESULTS[map_name]["model_summary"] for map_name in TARGET_MAPS if map_name in MAP_RESULTS],
    ignore_index=True,
)
display(combined_model_summary)

latex_code = format_cross_map_latex(
    combined_model_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(latex_code)

paths = save_summary_tables(
    combined_model_summary,
    FIGURES_DIR,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(f"Saved CSV: {paths['csv']}")
print(f"Saved TeX: {paths['tex']}")

BUNDLE_DIR = FIGURES_DIR / f"smax_checkpoint_eval_{CHECKPOINT_CHOICE}_bundle"

bundle_paths = save_full_evaluation_bundle(
    MAP_RESULTS,
    BUNDLE_DIR,
    checkpoint_choice=CHECKPOINT_CHOICE,
    eval_episodes=EVAL_EPISODES,
    eval_seeds=EVAL_SEEDS,
    map_order=TARGET_MAPS,
    model_order=TARGET_MODELS,
)
print(f"Saved bundle dir: {bundle_paths['dir']}")
print(f"Saved seed summary: {bundle_paths['seed_summary']}")
print(f"Saved model summary: {bundle_paths['model_summary']}")


,map_name,constraint_key,constraint_label,model_label,mean_success_rate,std_success_rate,mean_reward,std_reward,mean_length,std_length,n_seeds,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,ippo,0.903125,0.070226,1.883958,7.916872e-02,18.806250,2.652539,5,5,64
1,3m,vanilla,vanilla,pci-ippo,0.975000,0.008558,1.970208,1.068915e-02,26.250000,2.056068,5,5,64
2,3m,vanilla,vanilla,mappo,0.875000,0.052987,1.856667,6.497529e-02,15.593750,0.944505,5,5,64
3,3m,vanilla,vanilla,pci-mappo,0.931250,0.028384,1.920625,3.452775e-02,30.665625,0.996000,5,5,64
4,3m,tar_w=2,tar_w=2,ippo,0.409375,0.067568,1.259167,9.713618e-02,19.715625,6.785024,5,5,64
5,3m,tar_w=2,tar_w=2,pci-ippo,0.856250,0.069351,1.825208,9.479567e-02,36.409375,1.621593,5,5,64
6,3m,tar_w=2,tar_w=2,mappo,0.556250,0.246459,1.462292,2.987000e-01,17.537500,4.178740,5,5,64
7,3m,tar_w=2,tar_w=2,pci-mappo,0.887500,0.033874,1.862917,4.528316e-02,39.493750,1.922593,5,5,64
8,2s3z,vanilla,vanilla,ippo,0.984375,0.034939,1.982120,3.998079e-02,54.150000,4.265777,5,5,64
9,2s3z,vanilla,vanilla,pci-ippo,0.987500,0.020373,1.986894,2.145534e-02,56.381250,6.268221,5,5,64


\begin{table}[htbp]
\vspace{-1em}
\caption{Test win rates (\%) on SMAX maps. $\pm$ denotes one standard deviation over available training seeds. The best results are highlighted in bold. The sign of \(\Delta\) and colored values indicate performance improvements (blue) or degradations (red) over their respective baselines.}
\label{tab:smax}
\centering
\resizebox{0.9\textwidth}{!}{%

\begin{tabular}{c c c c c c c c}
\toprule
Map & Target Constraint & IPPO & PCI-IPPO* & \(\Delta\) & MAPPO & PCI-MAPPO* & \(\Delta\)\\
\midrule
    3m & Vanilla & 90.3 \scriptsize$\pm$ 7.0 & \textbf{97.5} \scriptsize$\pm$ 0.9 & \textbf{\textcolor{blue}{7.2}} & 87.5 \scriptsize$\pm$ 5.3 & 93.1 \scriptsize$\pm$ 2.8 & \textbf{\textcolor{blue}{5.6}} \\
     & Hard & 40.9 \scriptsize$\pm$ 6.8 & 85.6 \scriptsize$\pm$ 6.9 & \textbf{\textcolor{blue}{44.7}} & 55.6 \scriptsize$\pm$ 24.6 & \textbf{88.8} \scriptsize$\pm$ 3.4 & \textbf{\textcolor{blue}{33.1}} \\
    \midrule
    2s3z & Vanilla & 98.4 \scriptsize$\pm$ 3.5

In [10]:
if not MAP_RESULTS:
    raise RuntimeError("Run the full evaluation cell first.")

combined_seed_summary = pd.concat(
    [MAP_RESULTS[map_name]["seed_summary"] for map_name in TARGET_MAPS if map_name in MAP_RESULTS],
    ignore_index=True,
)
display(combined_seed_summary)

seedwise_group_a = ["3m", "2s3z", "3s5z"]
seedwise_group_b = ["smacv2_5_units", "smacv2_10_units", "6h_vs_8z"]

seedwise_latex_a = scu.format_seedwise_cross_map_latex(
    combined_seed_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=seedwise_group_a,
    model_order=TARGET_MODELS,
    caption=(
        r"Seed-wise deterministic SMAX test win rates (\%) using final checkpoints for "
        r"\texttt{3m}, \texttt{2s3z}, and \texttt{3s5z}. Each entry is the mean over 64 evaluation seeds; "
        "all configured training seeds are present."
    ),
    label="tab:smax_seedwise_part1",
)
print(seedwise_latex_a)

seedwise_latex_b = scu.format_seedwise_cross_map_latex(
    combined_seed_summary,
    checkpoint_choice=CHECKPOINT_CHOICE,
    map_order=seedwise_group_b,
    model_order=TARGET_MODELS,
    caption=(
        r"Seed-wise deterministic SMAX test win rates (\%) using final checkpoints for "
        r"\texttt{smacv2\_5\_units}, \texttt{smacv2\_10\_units}, and \texttt{6h\_vs\_8z}. "
        "Each entry is the mean over 64 evaluation seeds; all configured training seeds are present."
    ),
    label="tab:smax_seedwise_part2",
)
print(seedwise_latex_b)

seedwise_latex_a_path = FIGURES_DIR / f"smax_checkpoint_eval_{CHECKPOINT_CHOICE}_seedwise_part1.tex"
seedwise_latex_b_path = FIGURES_DIR / f"smax_checkpoint_eval_{CHECKPOINT_CHOICE}_seedwise_part2.tex"
seedwise_latex_a_path.write_text(seedwise_latex_a)
seedwise_latex_b_path.write_text(seedwise_latex_b)
print(f"Saved TeX A: {seedwise_latex_a_path}")
print(f"Saved TeX B: {seedwise_latex_b_path}")


,map_name,constraint_key,constraint_label,model_label,seed,eval_success_rate,eval_mean_reward,eval_mean_length,n_runs,n_eval_seeds
0,3m,vanilla,vanilla,ippo,1,0.875000,1.863542,19.156250,1,64
1,3m,vanilla,vanilla,ippo,2,0.890625,1.863542,15.906250,1,64
2,3m,vanilla,vanilla,ippo,31,0.937500,1.909375,21.843750,1,64
3,3m,vanilla,vanilla,ippo,32,1.000000,2.000000,20.828125,1,64
4,3m,vanilla,vanilla,ippo,42,0.812500,1.783333,16.296875,1,64
...,...,...,...,...,...,...,...,...,...,...
295,smacv2_10_units,tar_w=8,tar_w=8,pci-mappo,1,0.625000,1.516407,45.296875,1,64
296,smacv2_10_units,tar_w=8,tar_w=8,pci-mappo,2,0.609375,1.500609,44.468750,1,64
297,smacv2_10_units,tar_w=8,tar_w=8,pci-mappo,31,0.546875,1.415772,42.765625,1,64
298,smacv2_10_units,tar_w=8,tar_w=8,pci-mappo,32,0.609375,1.497938,42.734375,1,64


\begin{table*}[t]
\centering
\small
\setlength{\tabcolsep}{4pt}
\caption{Seed-wise deterministic SMAX test win rates (\%) using final checkpoints for \texttt{3m}, \texttt{2s3z}, and \texttt{3s5z}. Each entry is the mean over 64 evaluation seeds; all configured training seeds are present.}
\label{tab:smax_seedwise_part1}
\begin{tabular}{llccccc}
\toprule
Map & Constraint & Seed & IPPO & PCI-IPPO & MAPPO & PCI-MAPPO \\
\midrule
3m & vanilla & 1 & 87.5 & 96.9 & 90.6 & 90.6 \\
 &  & 2 & 89.1 & 98.4 & 79.7 & 92.2 \\
 &  & 31 & 93.8 & 98.4 & 84.4 & 90.6 \\
 &  & 32 & 100.0 & 96.9 & 90.6 & 95.3 \\
 &  & 42 & 81.2 & 96.9 & 92.2 & 96.9 \\
 & tar_w=2 & 1 & 40.6 & 92.2 & 50.0 & 89.1 \\
 &  & 2 & 42.2 & 87.5 & 39.1 & 84.4 \\
 &  & 31 & 45.3 & 75.0 & 98.4 & 89.1 \\
 &  & 32 & 29.7 & 82.8 & 39.1 & 93.8 \\
 &  & 42 & 46.9 & 90.6 & 51.6 & 87.5 \\
\midrule
2s3z & vanilla & 1 & 100.0 & 100.0 & 100.0 & 100.0 \\
 &  & 2 & 100.0 & 95.3 & 100.0 & 100.0 \\
 &  & 31 & 100.0 & 100.0 & 100.0 & 100.0 \\
 &  & 32